# Verity — Fake News Prediction

This notebook is the reproducible training and evaluation companion for the **React + FastAPI** application. It uses the **same dataset builder, preprocessing function, TF-IDF vectorizer, Logistic Regression settings, and exported model artifact** as the API.

**Labels:** `0 = Real News`, `1 = Fake News`  
**Features:** `title + text` (headline plus full article body)  
**Dataset:** ISOT Fake and Real News, a historical English-language benchmark.

> The ISOT corpus is largely focused on US politics and world news from 2016–2017. Its labels are source-derived and a random hold-out score is not proof of truthfulness, current-news accuracy, or generalization to unseen publishers. Treat model output as a screening signal, not a fact-check.

## 1. Locate the project and import the shared pipeline

Run this notebook from the project root or its `notebooks/` folder. The code searches parent folders for `backend/train_model.py`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

_current = Path.cwd().resolve()
_candidates = [_current, *_current.parents]
PROJECT_ROOT = next((p for p in _candidates if (p / "backend" / "train_model.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find backend/train_model.py. Open this notebook inside the extracted project folder."
    )
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from train_model import train, DEFAULT_DATA, DEFAULT_MODEL
from app.preprocessing import stemming

DATA_PATH = BACKEND_DIR / "data" / "train.csv"
MODEL_PATH = BACKEND_DIR / "models" / "fake_news_model.joblib"
print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Model output:", MODEL_PATH)

## 2. Prepare the combined dataset

The project includes your uploaded `fake.csv.zip`. The matching real-news file is retrieved from the University of Victoria ISOT archive when you run the builder with an internet connection. Alternatively, place the official `True.csv` in `backend/data/` first.

The builder creates `backend/data/train.csv` and `dataset_info.json`. It does not invent or synthesize real-news records.

In [ ]:
if not DATA_PATH.exists():
    import subprocess
    subprocess.run(
        [sys.executable, str(BACKEND_DIR / "prepare_dataset.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset was not created: {DATA_PATH}")

dataset = pd.read_csv(DATA_PATH, keep_default_na=False)
print("Dataset shape:", dataset.shape)
display(dataset.head())

## 3. Inspect dataset quality and class balance

In [ ]:
info_path = BACKEND_DIR / "data" / "dataset_info.json"
if info_path.exists():
    dataset_info = json.loads(info_path.read_text(encoding="utf-8"))
    display(pd.DataFrame({"item": list(dataset_info.keys()), "value": [str(v) for v in dataset_info.values()]}))

print("Missing values:")
display(dataset.isna().sum().rename("missing_values").to_frame())
print("Class counts (0=Real, 1=Fake):")
display(dataset["label"].value_counts().sort_index().rename(index={0:"Real News", 1:"Fake News"}).to_frame("articles"))

## 4. Train and evaluate the exact model used by FastAPI

The training function performs a stratified 80/20 split **before fitting TF-IDF** to avoid vocabulary leakage. It writes the model to `backend/models/fake_news_model.joblib` and its evaluation summary to `backend/models/metrics.json`.

In [ ]:
metrics = train(DATA_PATH, MODEL_PATH)
print(f"Accuracy:       {metrics['accuracy']:.4f}")
print(f"Fake precision: {metrics['precision_fake']:.4f}")
print(f"Fake recall:    {metrics['recall_fake']:.4f}")
print(f"Fake F1-score:  {metrics['f1_fake']:.4f}")
print(f"Fake ROC-AUC:   {metrics['roc_auc_fake']:.4f}")
print(f"Training rows:  {metrics['training_rows']:,}")
print(f"Test rows:      {metrics['test_rows']:,}")
print(f"Vocabulary:     {metrics['vocabulary_size']:,}")

## 5. Classification report

In [ ]:
report = pd.DataFrame(metrics["classification_report"]).T
display(report.round(3))

## 6. Confusion matrix

Rows are actual classes and columns are predicted classes. Class order is `0 = Real News`, `1 = Fake News`.

In [ ]:
import matplotlib.pyplot as plt

cm = metrics["confusion_matrix_labels_0_real_1_fake"]
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])
for row in range(2):
    for col in range(2):
        ax.text(col, row, str(cm[row][col]), ha="center", va="center")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## 7. Try a sample prediction using the saved artifact

This calls the same predictor function used by the FastAPI route. Change the headline and body to experiment with your own examples.

In [ ]:
from app.predictor import predict_news, load_artifact

load_artifact.cache_clear()  # ensure the just-trained artifact is loaded
sample_title = "Local council announces a new public library"
sample_text = "The city council approved funding for a public library and community reading program."
sample_result = predict_news(sample_title, sample_text)
print(json.dumps(sample_result, indent=2))

## 8. Run the web application

Open two terminals from the project root.

**Terminal 1 — FastAPI**
```bash
uvicorn app.main:app --reload --app-dir backend --host 127.0.0.1 --port 8000
```

**Terminal 2 — React**
```bash
cd frontend
npm install
npm run dev
```

Open `http://localhost:5173`. API documentation is available at `http://127.0.0.1:8000/docs`.

## 9. Limitations and responsible use

- The ISOT benchmark is historical and English-language; it is not a balanced sample of all current journalism.
- Its truthful articles came from Reuters, while fake articles came from sources classified as unreliable by the dataset authors. A model may learn source or writing-style differences rather than factual truth.
- A random train/test split measures performance on a held-out portion of this dataset, not on new publishers, languages, or current events.
- Logistic Regression probabilities are model scores and are not automatically calibrated.
- Verify important claims with primary sources and independent fact-checkers. Do not use this model as the sole basis for high-impact decisions.